### Customers — dedup mapping, missing/placeholder city fill, city formatting, invalid birth_date flag, dedup

In [0]:
%sql
-- Step 1: build canonical customer mapping to collapse duplicate person records
CREATE OR REPLACE TABLE ecommerce.stage.customer_dedup_map AS
WITH normalized AS (
  SELECT
    customer_id,
    TRIM(first_name) AS first_name,
    TRIM(last_name)  AS last_name,
    CASE WHEN city IS NULL OR LOWER(TRIM(city)) IN ('n/a','unknown','-','') THEN 'Unknown'
         ELSE INITCAP(TRIM(city)) END AS city_norm,
    birth_date
  FROM ecommerce.base.customers
)
SELECT
  customer_id,
  MIN(customer_id) OVER (PARTITION BY first_name, last_name, city_norm, birth_date) AS canonical_customer_id
FROM normalized;

In [0]:
%sql
-- Step 2: build clean.customers — keep only canonical record per person,
-- fill missing/placeholder city with 'Unknown', standardize casing, flag invalid birth_date
CREATE OR REPLACE TABLE ecommerce.clean.customers AS
SELECT
  c.customer_id,
  c.first_name,
  c.last_name,
  c.gender,
  c.birth_date,
  CASE WHEN c.birth_date > CURRENT_DATE OR c.birth_date < DATE '1900-01-01'
       THEN TRUE ELSE FALSE END AS is_invalid_birth_date,
  c.signup_date,
  CASE WHEN c.city IS NULL OR LOWER(TRIM(c.city)) IN ('n/a','unknown','-','')
       THEN 'Unknown'
       ELSE INITCAP(TRIM(c.city)) END AS city,
  c.region_id,
  c.customer_segment,
  c.acquisition_channel
FROM ecommerce.base.customers c
JOIN ecommerce.stage.customer_dedup_map m
  ON c.customer_id = m.customer_id
WHERE c.customer_id = m.canonical_customer_id;

### Products — fill missing brand

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.clean.products AS
SELECT
  product_id,
  product_name,
  category_id,
  COALESCE(brand, 'Unknown') AS brand,
  unit_cost,
  list_price,
  launch_date,
  product_status
FROM ecommerce.base.products;

### Orders — fill missing shipping_method

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.clean.orders AS
SELECT
  order_id,
  customer_id,
  order_date,
  region_id,
  store_id,
  order_status,
  COALESCE(shipping_method, 'Unknown') AS shipping_method,
  shipping_cost,
  discount_amount,
  tax_amount,
  order_total
FROM ecommerce.base.orders;

### Order Items — pass-through (duplicates deliberately retained as legitimate distinct line items)

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.clean.order_items AS
SELECT
  order_item_id,
  order_id,
  product_id,
  quantity,
  unit_price,
  discount_percent,
  discount_amount,
  item_revenue,
  unit_cost,
  total_cost,
  gross_profit
FROM ecommerce.base.order_items;

### Payments — standardize payment_method formatting, flag payment_date anomaly

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.clean.payments AS
SELECT
  p.payment_id,
  p.order_id,
  p.payment_date,
  CASE WHEN p.payment_date < o.order_date THEN TRUE ELSE FALSE END AS is_payment_date_anomaly,
  INITCAP(TRIM(p.payment_method)) AS payment_method,
  p.payment_status,
  p.amount
FROM ecommerce.base.payments p
LEFT JOIN ecommerce.base.orders o ON p.order_id = o.order_id;

### Returns — standardize return_reason formatting

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.clean.returns AS
SELECT
  return_id,
  order_id,
  return_date,
  UPPER(TRIM(return_reason)) AS return_reason,
  refund_amount
FROM ecommerce.base.returns;

## Post-Cleaning Verification 

### Payment date anomaly — now flagged, not removed

In [0]:
%sql
-- Row count should be unchanged (nothing deleted);
SELECT
  COUNT(*) AS total_payments,
  SUM(CASE WHEN is_payment_date_anomaly THEN 1 ELSE 0 END) AS flagged_anomaly_count
FROM ecommerce.clean.payments;

### Missing values resolved — city, brand, shipping_method

In [0]:
%sql
SELECT 'clean.customers.city' AS field, COUNT(*) - COUNT(city) AS null_count
FROM ecommerce.clean.customers
UNION ALL
SELECT 'clean.products.brand', COUNT(*) - COUNT(brand)
FROM ecommerce.clean.products
UNION ALL
SELECT 'clean.orders.shipping_method', COUNT(*) - COUNT(shipping_method)
FROM ecommerce.clean.orders;
-- Expect 0 for all three

### Duplicate order_items — confirm unchanged (kept intentionally)

In [0]:
%sql
SELECT COUNT(*) AS duplicate_groups
FROM (
  SELECT order_id, product_id, quantity, unit_price
  FROM ecommerce.clean.order_items
  GROUP BY order_id, product_id, quantity, unit_price
  HAVING COUNT(*) > 1
);
-- Expect same count as Section 1.3 — no rows removed

### Payment method formatting — should now be a small, consistent set

In [0]:
%sql
SELECT payment_method, COUNT(*) AS occurrences
FROM ecommerce.clean.payments
GROUP BY payment_method
ORDER BY occurrences DESC;

###  City formatting — variations should collapse to a single spelling each

In [0]:
%sql
SELECT
    LOWER(city) AS standardized_city,
    COUNT(DISTINCT city) AS raw_variations,
    COUNT(*) AS total_rows
FROM ecommerce.clean.customers
GROUP BY LOWER(city)
HAVING COUNT(DISTINCT city) > 1
ORDER BY raw_variations DESC;
-- Expect 0 rows returned

### Invalid birth_date — now flagged

In [0]:
%sql
SELECT
  COUNT(*) AS total_customers,
  SUM(CASE WHEN is_invalid_birth_date THEN 1 ELSE 0 END) AS flagged_invalid_birth_dates
FROM ecommerce.clean.customers;

###  Return reason formatting — should now be a small, consistent set

In [0]:
%sql
SELECT return_reason, COUNT(*) AS occurrences
FROM ecommerce.clean.returns
GROUP BY return_reason
ORDER BY occurrences DESC;

###  Duplicate customer records — row count reduced, no duplicate groups remain

In [0]:
%sql
-- Row count comparison
SELECT
  (SELECT COUNT(*) FROM ecommerce.base.customers)  AS base_row_count,
  (SELECT COUNT(*) FROM ecommerce.clean.customers) AS clean_row_count;

In [0]:
%sql
-- Confirm no duplicate groups remain in clean.customers
SELECT first_name, last_name, city, birth_date, COUNT(*) AS occurrences
FROM ecommerce.clean.customers
GROUP BY first_name, last_name, city, birth_date
HAVING COUNT(*) > 1;
-- Expect 0 rows

###  Placeholder city values — should no longer exist (replaced with 'Unknown')

In [0]:
%sql
SELECT city, COUNT(*) AS occurrences
FROM ecommerce.clean.customers
WHERE LOWER(city) IN ('n/a', '-', '')
GROUP BY city;
-- Expect 0 rows. 'Unknown' will legitimately still appear (that's the intended fill value).